# MECオフロード比較：起動エネルギー 1.5 J（ja）

共通処理は `analysis/reporting.py` にまとめています。設定 → 読込 → 図 → 表の順に実行してください。
基準CSVとシミュレーションコードは変更しません。


## 1. 設定

`DATA_DIR` で入力先を指定します。`SHOW_ALTERNATIVES = True` にすると、既存指標の別表示も作成します。


In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = next(
    (p for p in (Path.cwd(), *Path.cwd().parents)
     if (p / "configs/boot_1.0.json").is_file() and (p / "analysis/reporting.py").is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError("研究フォルダまたは analysis フォルダから開いてください。")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from analysis.reporting import ExperimentData, AnalysisReport

CONDITION = 3
LANGUAGE = "ja"
DATA_DIR = PROJECT_ROOT / "data/reference"
OUTPUT_DIR = PROJECT_ROOT / "outputs/analysis/organized/analyze03"
FORMATS = ("pdf",)
INCLUDE_RUNTIME = False  # 元のノートブックの出力範囲を維持
SHOW_ALTERNATIVES = False


## 2. CSVを一度だけ読み込む

集計CSVとユーザ別CSVを読み込みます。実行時間の2手法比較でも、このデータを絞り込み直しません。


In [ ]:
data = ExperimentData.load(DATA_DIR, CONDITION)
report = AnalysisReport(data, OUTPUT_DIR, language=LANGUAGE, formats=FORMATS)
print(f"集計: {len(data.runs)} 行、ユーザ別: {len(data.users)} 行")
print(f"出力先: {OUTPUT_DIR}")


## 3. エネルギー・遅延・公平性と起動MEC台数

連続値の8指標はバイオリン図、起動MEC台数は箱ひげ図を1回ずつ出力します。
締切違反率はCSVの値をそのまま使用します（実装上の注意点はルートREADME参照）。


In [ ]:
report.plot_metrics()


## 4. ユーザ別の遅延

総遅延は全シナリオ・全ユーザの合計です。締切超過時間の箱ひげ図は `excess > 0` のユーザだけを対象とします。


In [ ]:
report.plot_user_delays()


## 5. 実行時間と任意の別表示

実行時間の標準表示は4手法の対数軸バイオリン図です。任意表示には起動台数のバイオリン図、実行時間の平均棒グラフ、GA対CEGAの線形軸バイオリン図があります。


In [ ]:
if INCLUDE_RUNTIME:
    report.plot_runtime()
if SHOW_ALTERNATIVES:
    report.plot_alternatives(include_runtime=INCLUDE_RUNTIME)


## 6. 集計表

エネルギーとQoEの平均・母分散（`ddof=0`）、必要なら実行時間の平均を各1回表示します。
同じ集計からCSVとLaTeXを保存します。LaTeXは小数点以下6桁です。


In [ ]:
tables = report.write_tables(include_runtime=INCLUDE_RUNTIME)
for name, table in tables.items():
    print(name)
    display(table.rename(index={"GA": "FBE-CEGA"}))
print(f"図: {len(report.generated)} ファイル、表: {len(tables)} 種類（CSV / TeX）")
